In [ ]:
# Required installations (you may need a specific LangChain version for IBM integration)
# pip install ragas datasets langchain_ibm ibm-watson-machine-learning

import os
from datasets import Dataset
from ragas import evaluate # https://docs.ragas.io/en/stable/
from ragas.metrics import faithfulness, answer_relevancy, context_precision, NoiseSensitivity

# Import the necessary IBM client from LangChain integration
from langchain_ibm import WatsonxLLM 
# Note: RAGAS typically uses the RAGAS LLM base class, 
# but often integrates via LangChain wrappers if a direct factory is not available.

In [38]:
# --- 1. Initialize the IBM Granite LLM Client ---
# Set your environment variables (e.g., in your shell or script)
# os.environ["WATSONX_APIKEY"] = "YOUR_WATSONX_API_KEY"
# os.environ["WATSONX_PROJECT_ID"] = "YOUR_PROJECT_ID"

WATSONX_APIKEY = os.environ.get("WATSONX_APIKEY", "YOUR_API_KEY") 
WATSONX_PROJECT_ID = os.environ.get("IBM_PROJECT_ID", "YOUR_PROJECT_ID")

# Define model parameters (often required for Granite models)
parameters = {
    "decoding_method": "sample",
    "max_new_tokens": 5000, # Max tokens for the LLM-as-a-Judge response
    "temperature": 0.0,    # Use low temperature for deterministic evaluation
}

# Create the WatsonxLLM client instance for the RAGAS evaluation
# RAGAS will use this LLM to run its metrics.
granite_llm_client = WatsonxLLM(
    model_id="ibm/granite-3-2-8b-instruct", # Example Granite model ID
    url="https://us-south.ml.cloud.ibm.com", # Update with your region endpoint
    project_id=WATSONX_PROJECT_ID,
    params=parameters
)
# Note: RAGAS often requires separate embeddings setup for metrics like Answer Relevancy.
# For simplicity, this example relies on RAGAS's default embedding setup (which may use HuggingFace or require explicit setting).

c:\ProgramData\anaconda3\envs\docling\Lib\site-packages\ibm_watsonx_ai\foundation_models\utils\utils.py:431: LifecycleWarning: Model 'ibm/granite-3-2-8b-instruct' is in deprecated state from 2025-11-24 until 2026-02-22. IDs of alternative models: ibm/granite-4-h-small. Further details: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-model-lifecycle.html?context=wx&audience=wdp
  warn(model_state_warning, category=LifecycleWarning)


In [ ]:
# --- 2. Prepare BAD Demo Evaluation Dataset (The test data) ---
data = {
    'question': [
        "What is the capital of France and what is the key landmark there?",
    ],
    'answer': [
        "The capital of France is Paris and it has many excellent restaurants and shopping centres",
    ],
    'contexts': [
        [
            "France is a developed country in Western Europe.", 
            "Paris is its most populous city and city of governance.", 
            "The French is a producer of wine and cheese.",
        ],
    ],
    'reference': [
        "Paris, with the Eiffel Tower as its most famous monument."
    ],
}
dataset = Dataset.from_dict(data)
# --- 3. Define the Metrics ---
metrics = [faithfulness, NoiseSensitivity(), context_precision]

# --- 4. Run the Evaluation ---
print("Starting RAG evaluation with IBM Granite...")
result = evaluate(
    dataset,
    metrics=metrics,
    # Pass the configured IBM Granite client
    llm=granite_llm_client
)

# --- 5. Print the Results ---
print("\n--- Evaluation Scores ---")
print(result)

# Accessing individual scores
print("\n--- Summary Statistics ---")
print(f"Overall Faithfulness Score: {result['faithfulness']}")
print(f"Overall Noise Sensitivity Score: {result['noise_sensitivity(mode=relevant)']}")
print(f"Overall Context Precision Score: {result['context_precision']}")

Starting RAG evaluation with IBM Granite...


Evaluating: 100%|██████████| 3/3 [00:41<00:00, 13.98s/it]



--- Evaluation Scores ---
{'faithfulness': 0.5000, 'noise_sensitivity(mode=relevant)': 1.0000, 'context_precision': 0.0000}

--- Summary Statistics ---
Overall Faithfulness Score: [0.5]
Overall Answer Relevancy Score: [1.0]
Overall Context Precision Score: [0.0]


In [45]:
# --- 2. Prepare GOOD Demo Evaluation Dataset (The test data) ---
data = {
    'question': [
        "What is the capital of France and what is the key landmark there?",
    ],
    'answer': [
        "The capital of France is Paris, and its most famous landmark is the Eiffel Tower.",
    ],
    'contexts': [
        [
            "Paris is the capital and most populous city of France. The Eiffel Tower, a wrought-iron lattice tower on the Champ de Mars, is the city's most visited paid monument.",
            "The Louvre Museum is another major Parisian landmark, known for housing the Mona Lisa.",
            "France is famous for its wines and cuisine."
        ],
    ],
    'reference': [
        "Paris, with the Eiffel Tower as its most famous monument."
    ],
}
dataset = Dataset.from_dict(data)
# --- 3. Define the Metrics ---
metrics = [faithfulness, NoiseSensitivity(), context_precision]

# --- 4. Run the Evaluation ---
print("Starting RAG evaluation with IBM Granite...")
result = evaluate(
    dataset,
    metrics=metrics,
    # Pass the configured IBM Granite client
    llm=granite_llm_client
)

# --- 5. Print the Results ---
print("\n--- Evaluation Scores ---")
print(result)

# Accessing individual scores
print("\n--- Summary Statistics ---")
print(f"Overall Faithfulness Score: {result['faithfulness']}")
print(f"Overall Noise Sensitivity Score: {result['noise_sensitivity(mode=relevant)']}")
print(f"Overall Context Precision Score: {result['context_precision']}")

Starting RAG evaluation with IBM Granite...


Evaluating: 100%|██████████| 3/3 [00:53<00:00, 17.92s/it]



--- Evaluation Scores ---
{'faithfulness': 1.0000, 'noise_sensitivity(mode=relevant)': 0.0000, 'context_precision': 1.0000}

--- Summary Statistics ---
Overall Faithfulness Score: [1.0]
Overall Noise Sensitivity Score: [0.0]
Overall Context Precision Score: [0.9999999999]
